# PVS Demo Pipeline

This notebook demonstrates a basic workflow using the Prime Vector Space (PVS) library.
It covers:
1. Generating a prime vector.
2. Performing a conceptual analysis on the generated vector.
3. Visualizing some aspects of the data (if `matplotlib` is installed).

## 1. Setup and Imports

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os
from pathlib import Path

# Add the project root to the Python path to import pvs library
# This assumes the notebook is in 'notebooks/' and the library in 'pvs/' at the project root.
project_root = Path.cwd().parent # Assuming current dir is 'notebooks'
sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd

from pvs.cli import generate, analyse # Using CLI functions directly for simplicity
# Alternatively, import from library components:
# from pvs.geometry import build_prime_vector
# from pvs.weights import gpy_weight

# For plotting (optional)
try:
    import matplotlib.pyplot as plt
    PLOT_AVAILABLE = True
except ImportError:
    PLOT_AVAILABLE = False
    print("Matplotlib not found. Plotting will be skipped. Install with: pip install matplotlib or poetry install --extras viz")

# Ensure data directories exist (similar to CLI's ensure_data_dir)
data_dir = project_root / "data"
raw_data_dir = data_dir / "raw"
processed_data_dir = data_dir / "processed"
raw_data_dir.mkdir(parents=True, exist_ok=True)
processed_data_dir.mkdir(parents=True, exist_ok=True)

print(f"Project root: {project_root}")
print(f"Processed data directory: {processed_data_dir}")

## 2. Generate Prime Vector

We'll use the `generate` functionality (similar to `pvs generate` CLI command) to create a prime vector file.

In [ ]:
MAX_N = 1000  # Generate primes up to 1000
OUTPUT_FILENAME = f"demo_prime_vector_{MAX_N}.parquet"
output_path = processed_data_dir / OUTPUT_FILENAME

print(f"Generating prime vector for n <= {MAX_N}...")
# We can invoke the Click command functions programmatically if needed,
# but it's often cleaner to call the underlying library functions directly in a notebook.

# Using the CLI function's logic directly (simplified for notebook use):
from pvs.geometry import build_prime_vector

prime_vector_sparse = build_prime_vector(MAX_N)
prime_indices = prime_vector_sparse.indices
df_primes = pd.DataFrame({'prime_indices': prime_indices})

try:
    df_primes.to_parquet(output_path)
    print(f"Prime vector saved to: {output_path}")
except Exception as e:
    print(f"Error saving Parquet file: {e}")

# Display some info about the generated data
print(f"Number of primes found up to {MAX_N}: {len(df_primes)}")
if not df_primes.empty:
    print("First few prime indices:")
    print(df_primes.head())
    print("Last few prime indices:")
    print(df_primes.tail())

## 3. Analyse Prime Vector

Now, let's use the `analyse` functionality (similar to `pvs analyse` CLI command).

In [ ]:
if not output_path.exists():
    print(f"Error: Parquet file {output_path} not found. Please run step 2 first.")
else:
    # Parameters for analysis
    K_PARAM = 2  # Example: for twin primes
    U_FACTOR_PARAM = 0.5

    print(f"\nAnalysing prime vector from: {output_path}")
    print(f"Parameters: k={K_PARAM}, u-factor={U_FACTOR_PARAM}")

    # Replicating logic from cli.analyse for demonstration
    from pvs.weights import gpy_weight

    df_loaded_primes = pd.read_parquet(output_path)
    loaded_prime_indices = df_loaded_primes['prime_indices'].values
    max_n_from_file = loaded_prime_indices[-1] if len(loaded_prime_indices) > 0 else 0

    if max_n_from_file <= 1:
        u_param = 1.0
    else:
        u_param = U_FACTOR_PARAM * np.log(max_n_from_file)
    if u_param <=0: u_param = 1.0

    try:
        weights = gpy_weight(K_PARAM, u_param)
        print(f"Computed GPY weights (k={K_PARAM}, u={u_param:.2f}): {weights}")

        # Placeholder theta statistic
        theta_stat = np.sum(weights) * np.mean(loaded_prime_indices) if len(loaded_prime_indices) > 0 else 0
        print(f"Placeholder θ angle statistic: {theta_stat:.4f}")

        if PLOT_AVAILABLE:
            print("\nGenerating plots...")
            fig, ax = plt.subplots(1, 2, figsize=(12, 5))

            # Plot 1: Prime distribution
            if len(loaded_prime_indices) > 0:
                ax[0].hist(loaded_prime_indices, bins=min(50, int(np.sqrt(len(loaded_prime_indices)))*2 if len(loaded_prime_indices)>0 else 1), color='skyblue', edgecolor='black')
                ax[0].set_title(f"Distribution of Primes (up to {max_n_from_file})")
                ax[0].set_xlabel("Prime Number")
                ax[0].set_ylabel("Frequency (in bins)")
            else:
                ax[0].text(0.5, 0.5, "No prime data to plot.", ha='center', va='center', transform=ax[0].transAxes)
                ax[0].set_title("Prime Distribution")

            # Plot 2: GPY Weights
            ax[1].bar(range(len(weights)), weights, color='lightgreen', edgecolor='black')
            ax[1].set_title(f"Conceptual GPY Weights (k={K_PARAM}, u={u_param:.2f})")
            ax[1].set_xlabel("Weight Index")
            ax[1].set_ylabel("Weight Value")
            ax[1].set_xticks(range(len(weights)))

            plt.tight_layout()
            plot_filename = processed_data_dir / f"demo_analysis_plot_n{MAX_N}_k{K_PARAM}.png"
            plt.savefig(plot_filename)
            print(f"Plot saved to {plot_filename}")
            plt.show()
        else:
            print("Plotting skipped as matplotlib is not available.")

    except ValueError as e:
        print(f"Error during analysis: {e}")
    except Exception as e:
        print(f"An unexpected error occurred during analysis: {e}")

## 4. Further Exploration

- Try different values for `MAX_N`, `K_PARAM`, and `U_FACTOR_PARAM`.
- Extend the `gpy_weight` function in `pvs/weights.py` for more realistic calculations.
- Implement actual θ angle statistics in `pvs/cli.py` or a new analysis module.
- Explore larger values of `MAX_N` and observe performance (the `build_prime_vector` function has a stub for `n > 1e8`).